# 59. 새 압력 전환 신호의 방향 확인
58번 누락가스01:57:47을본뒤고른4개방향가설이다. 테스트영향이있는추가탐색이며독립검증은아니다. 압력하락/속도상승/전환압더큰하락/쿠션상승을확인. 실제압력전달실패·배기원인정답이없어단정하지않는다.
전문가경계는학습내부정상99%로고정. 57다수결앵커와54탐색최고앵커를구분해비교한다. 내부에서효과를요구한선택도별도공개한다. 부동소수점오차를변화로보지않도록압력·속도차이소수2자리/쿠션3자리로반올림후방향판정한다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
source=''.join(json.loads((root/'notebooks/58_rg3_missed11_dynamic_experts.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(source.rsplit('records=[]',1)[0])
# Use sign of rounded sensor differences to avoid interpreting floating-point noise as a change.
dp=np.round(d.Max_Injection_Pressure__delta1.to_numpy(),2)
ds=np.round(d.Max_Switch_Over_Pressure__delta1.to_numpy(),2)
dv=np.round(d.Max_Injection_Speed__delta1.to_numpy(),2)
dc=np.round(d.Cushion_Position__delta1.to_numpy(),3)
gates={
 'pressure_fall_speed_rise':(dp<0)&(dv>0),
 'pressure_fall_switch_steeper':(dp<0)&(ds<dp),
 'pressure_fall_speed_rise_switch_steeper':(dp<0)&(dv>0)&(ds<dp),
 'pressure_fall_speed_rise_cushion_rise':(dp<0)&(dv>0)&(dc>0)}
anchors={
 'majority57':pd.read_csv(tables/'rg3_pdf_fixed_majority_predictions.csv',parse_dates=['TimeStamp']).set_index(['scheme','TimeStamp']),
 'exploratory54':pd.read_csv(tables/'rg3_pdf_confirm_gates_predictions.csv',parse_dates=['TimeStamp']).query("model == 'thermal_change_low_0.05'").set_index(['scheme','TimeStamp'])}
records=[];choices=[];audit=[];innerrows=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
 for j,b in enumerate(blocks):
  q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
  if len(np.unique(y[ref]))<2:continue
  outer=predict(ref,q,'pressure_transition')[:,1]
  inner=np.full(len(ref),np.nan)
  for ib in groups.iloc[ref].unique():
   mask=groups.iloc[ref].eq(ib).to_numpy()
   inner[mask]=predict(ref[~mask],ref[mask],'pressure_transition')[:,1]
  ic=oldinner[(oldinner.scheme==scheme)&(oldinner.heldout_block==b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
  cb=float(np.quantile(ic.loc[y[ref]==0,'raw_baseline_score'],.99,method='higher'))
  oldpr=ic.raw_baseline_score.gt(cb).to_numpy()
  tn0,fp0,fn0,tp0=confusion_matrix(y[ref],oldpr,labels=[0,1]).ravel()
  f0=2*tp0/(2*tp0+fp0+fn0) if 2*tp0+fp0+fn0 else 0
  ns=(y[ref]==0)&np.isfinite(inner)
  cut=float(np.quantile(inner[ns],.99,method='higher'))
  anc={a:p.loc[(scheme,d.TimeStamp.iloc[q]),'prediction'].to_numpy(bool) for a,p in anchors.items()}
  local=[]
  for name,gate in gates.items():
   pr=oldpr|((inner>cut)&gate[ref])
   tn,fp,fn,tp=confusion_matrix(y[ref],pr,labels=[0,1]).ravel()
   fi=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   row=dict(scheme=scheme,block=b,gate=name,inner_added_TP=int(tp-tp0),inner_added_FP=int(fp-fp0),inner_F1=fi,allowed=bool(tp>tp0 and fp-fp0<=2 and fi>f0),train_gas_in_direction=int(gate[ref][y[ref]==1].sum()),train_normal_in_direction=int(gate[ref][y[ref]==0].sum()))
   innerrows.append(row);extra=(outer>cut)&gate[q];local.append((row,extra))
   for anchor_name,a in anc.items():
    pr=a|extra
    for l,i in enumerate(q):records.append(dict(scheme=scheme,anchor=anchor_name,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(pr[l]),anchor_prediction=int(a[l]),score=float(outer[l]),threshold=cut,direction_pass=int(gate[i])))
  valid=[c for c in local if c[0]['allowed']]
  best=max(valid,key=lambda c:(c[0]['inner_F1'],-c[0]['inner_added_FP'])) if valid else None
  pr=anc['majority57'] if best is None else anc['majority57']|best[1]
  choices.append(dict(scheme=scheme,block=b,selected='none' if best is None else best[0]['gate']))
  for l,i in enumerate(q):records.append(dict(scheme=scheme,anchor='majority57',model='inner_guarded_selector',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(pr[l]),anchor_prediction=int(anc['majority57'][l]),score=float(outer[l]),threshold=cut,direction_pass=np.nan))
  audit.append(dict(scheme=scheme,block=b,train=len(ref),test=len(q),timestamp_overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
  print('Completed pressure direction',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,anchor_name,model),g in pp.groupby(['scheme','anchor','model'],sort=False):
 assert not g.TimeStamp.duplicated().any()
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all()
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,anchor=anchor_name,model=model,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
mm=pd.DataFrame(rows)
for obj,name in [(mm,'metrics'),(pp,'predictions'),(pd.DataFrame(choices),'choices'),(pd.DataFrame(innerrows),'inner_candidates'),(pd.DataFrame(audit),'fold_audit')]:
 obj.to_csv(tables/f'rg3_pressure_direction_{name}.csv',index=False,encoding='utf-8-sig')
print(mm.to_string(index=False))
